In [1]:
import re
import os
import statistics
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np

PATTERN = re.compile(
    r'^(?P<timestamp>\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}-\d{2}:\d{2})\s+'
    r'\w+\s+ptp4l\[\d+\]:\s+ptp4l\[(?P<internal_ts>\d+\.\d+)\]:\s+'
    r'master\s+offset\s+(?P<offset>[+-]?\d+)\s+s\d\s+'
    r'freq\s+(?P<freq>[+-]?\d+)\s+'
    r'path\s+delay\s+(?P<delay>[+-]?\d+)'
)


def create_plot(data: list[float], device_name: str, plot_type: str):
    os.makedirs("plots", exist_ok=True)
    filename = f"plots/{device_name}-{plot_type}.png".lower()

    plt.figure(figsize=(8, 6))
    plt.plot(data, color="red")
    plt.title(f"{device_name} {plot_type}", fontsize=14)
    plt.xlabel("Sample Number")
    plt.ylabel("Value (nanoseconds)")
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()

def create_qq_plot(data: list[float], device_name: str, plot_type: str):
    os.makedirs("plots", exist_ok=True)
    filename = f"plots/{device_name}-{plot_type}-qq.png".lower()

    fig = sm.qqplot(np.array(data), line='45')
    plt.title(f"{device_name} {plot_type} — Q-Q Plot", fontsize=14)
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()

def parse_file(path: str, name: str):
    offsets, delays = [], []

    with open(path, "r") as f:
        for line in f:
            match = PATTERN.match(line.strip())
            if match:
                offsets.append(float(match.group("offset")))
                delays.append(float(match.group("delay")))

    if offsets and delays:
        for label, data in [("Offset", offsets), ("Delay", delays)]:
            print(f"\t{label} Stats:")
            print(f"\t  Mean: {statistics.mean(data):.2f}")
            print(f"\t  Min: {min(data):.2f}")
            print(f"\t  Max: {max(data):.2f}")
            print(f"\t  Std Dev: {statistics.stdev(data):.2f}")
            create_plot(data, name, label)
            create_qq_plot(data, name, label)
            if label == "Offset":
                print()
    else:
        print("No valid offset or delay data found.")

In [2]:
base_path = "/home/agebhard/Documents/repos/ptp-stats/data"
machines = [
    ("Beta",    os.path.join(base_path, "beta.log")),
    ("Charlie", os.path.join(base_path, "charlie.log")),
    ("Delta",   os.path.join(base_path, "delta.log")),
    ("Echo",    os.path.join(base_path, "echo.log")),
]

for name, path in machines:
    print(name)
    try:
        parse_file(path, name)
        print()
    except Exception as err:
        print(f"[{name}] Error: {err}")

Beta
	Offset Stats:
	  Mean: 0.04
	  Min: -206.00
	  Max: 199.00
	  Std Dev: 65.30

	Delay Stats:
	  Mean: 2917.65
	  Min: 2848.00
	  Max: 2985.00
	  Std Dev: 16.19

Charlie
	Offset Stats:
	  Mean: 0.00
	  Min: 0.00
	  Max: 0.00
	  Std Dev: 0.00

	Delay Stats:
	  Mean: 0.00
	  Min: 0.00
	  Max: 0.00
	  Std Dev: 0.00

Delta
	Offset Stats:
	  Mean: 0.21
	  Min: -229.00
	  Max: 233.00
	  Std Dev: 58.17

	Delay Stats:
	  Mean: 186.49
	  Min: 159.00
	  Max: 220.00
	  Std Dev: 4.66

Echo
	Offset Stats:
	  Mean: 0.08
	  Min: -206.00
	  Max: 218.00
	  Std Dev: 65.94

	Delay Stats:
	  Mean: 3114.25
	  Min: 3051.00
	  Max: 3183.00
	  Std Dev: 16.55

